# Finansal Sentiment Tezi - Çalışma Özeti

## 1. Çalışmanın Amacı

Bu çalışmada finansal haber ve kısa piyasa metinlerinin `negative`, `neutral` ve
`positive` olarak sınıflandırılması amaçlanmıştır. Hazır FinBERT modeli baseline
olarak kullanılmış; BERT, DistilBERT ve RoBERTa modelleri hedef veri seti üzerinde
fine-tune edilerek farklı test setlerinde karşılaştırılmıştır.

## Kullanılan Veri Kaynakları

### Plain Sentiment Eğitim Kaynakları

| Veri seti | Projede kullanılan satır | Temel özellik | Kullanım amacı |
| --- | ---: | --- | --- |
| [Twitter Financial News Sentiment](https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment) | 11.931 | İngilizce finans ve piyasa odaklı kısa metinler içerir. Etiketler bullish, bearish ve neutral piyasa yönelimini temsil eder. | Kısa piyasa metinlerine uygun ana eğitim verisi |
| [Financial PhraseBank](https://arxiv.org/abs/1307.5336) | 4.846 | Finans haberlerinden alınan İngilizce cümlelerden oluşur. Finans bilgisine sahip anotatörler tarafından positive, neutral ve negative sınıflarıyla etiketlenmiştir. | Finans haber dilini eğitim havuzuna ekleyen yardımcı veri |
| **Birleştirilmiş plain sentiment tablosu** | **16.777** | İki kaynak ortak üç sınıflı formata dönüştürülmüştür. | BERT, DistilBERT ve RoBERTa fine-tuning işlemleri |

### Target-Level Sentiment Eğitim Kaynakları

| Veri seti | Projede kullanılan satır | Temel özellik | Kullanım amacı |
| --- | ---: | --- | --- |
| [SEntFiN 1.0](https://arxiv.org/abs/2305.12257) | 14.371 | Haber başlıklarında şirket veya finansal varlık bazında sentiment anotasyonları içerir. Orijinal çalışmada 10.753 başlık ve birden fazla şirket içeren 2.847 başlık raporlanmıştır. | Şirket bazlı sentiment analizi |
| [FiQA 2018](https://sites.google.com/view/fiqa) | 1.173 | Finansal mikroblog ve haber başlıklarında sentiment ile aspect bilgisi sunan açık görev veri setidir. | Hedef ve aspect duyarlı örneklerin eklenmesi |
| [FinEntity](https://aclanthology.org/2023.emnlp-main.956/) | 979 | Finans haberlerinde entity span ve entity-level positive, neutral, negative anotasyonları içerir. | Belirli şirkete yönelen sentiment bilgisinin modellenmesi |
| **Birleştirilmiş target-level tablo** | **16.523** | Kaynaklar hedef bilgisi korunarak ortak formata getirilmiştir. | Target-based FinBERT eğitim akışı |

### Dış Test Kaynakları

| Veri seti | Projedeki kullanım | Temel özellik |
| --- | --- | --- |
| [S&P 500 with Financial News Headlines (2008-2024)](https://www.kaggle.com/datasets/dyutidasmahaptra/s-and-p-500-with-financial-news-headlines-20082024) | Ham kaynaktan 17.917 başlık işlendi; bağımsız dış test için 960 anotasyonlu başlık kullanıldı. | Günlük finans haber başlıklarını S&P 500 kapanış değerleriyle birlikte sunar. |
| Reuters haber alt kümesi | Yerel haber havuzundan Reuters başlıkları çıkarıldı ve 5.000 örneklik anotasyon seti hazırlandı. | Projede oluşturulan bağımsız dış test setidir; ayrı bir açık veri seti olarak sunulmamıştır. |
| Sentetik finans haberleri | Her sınıftan 1.000 örnek, toplam 3.000 haber kullanıldı. | Modellerin açık sentiment sinyallerindeki davranışını dengeli biçimde kontrol etmek için hazırlanmıştır. |

**Not:** Tablolardaki “projede kullanılan satır” değerleri yerel işlenmiş dosyalardan
alınmıştır. Kaynak veri setlerinin orijinal boyutları ile aynı olmak zorunda değildir;
özellikle target-level tabloda bir başlıktan birden fazla şirket bazlı örnek üretilebilir.

## 2. Hazırlanan Ana Veri Tabloları

| Tablo | Satır | Açıklama |
| --- | ---: | --- |
| `db/processed/aggregated_financial_news_enriched.csv` | 943.326 | Birleştirilmiş ve zenginleştirilmiş finans haber havuzu |
| `db/processed/market_sentiment_modeling_master.parquet` | 42.178 | Piyasa yönü modellemesi için hazırlanan ana tablo |
| `db/processed/sp500_headlines_2008_2024_finbert_labeled.csv` | 17.917 | FinBERT ile etiketlenen S&P 500 haber başlıkları |
| `db/interim/pseudo_labeled_news_confidence_090.parquet` | 88.342 | FinBERT güven skoru en az `0.90` olan pseudo-label haberler |
| `db/processed/training_datasets/plain_sentiment_dataset.parquet` | 16.777 | Genel sentiment sınıflandırması için eğitim tablosu |
| `db/processed/training_datasets/target_level_sentiment_dataset.parquet` | 16.523 | Hedef veya şirket bazlı sentiment analizi için tablo |

### Plain Sentiment Veri Dağılımı

| Sınıf | Adet | Oran |
| --- | ---: | ---: |
| Neutral | 10.623 | %63,32 |
| Positive | 3.761 | %22,42 |
| Negative | 2.393 | %14,26 |
| **Toplam** | **16.777** | **%100** |

Veri setinde `neutral` sınıfı baskındır. Bu nedenle model karşılaştırmalarında
yalnızca accuracy değil, sınıfları eşit ağırlıkla değerlendiren **macro-F1**
metriği de temel alınmıştır.

## 3. Uygulanan İşlem Sırası

| Sıra | Notebook | Yapılan işlem |
| ---: | --- | --- |
| 1 | `00_setup_requirements.ipynb` | Çalışma ortamı ve gerekli paketler hazırlandı. |
| 2 | `01a_build_reuters_annotation_dataset.ipynb` | Reuters haber başlıklarından 5.000 örneklik anotasyon seti oluşturuldu. |
| 3 | `01b_build_sp500_annotation_dataset.ipynb` | S&P 500 başlıklarından dengeli anotasyon örnekleri ve inceleme batchleri üretildi. |
| 4 | `02_evaluate_finbert_baseline.ipynb` | Hazır `ProsusAI/finbert` modeli baseline olarak ölçüldü ve hata analizi yapıldı. |
| 5 | `03_train_target_based_finbert.ipynb` | Şirket veya hedef bilgisi içeren target-level FinBERT eğitim akışı hazırlandı. |
| 6 | `04_train_plain_sentiment_models.ipynb` | BERT, DistilBERT ve RoBERTa modelleri plain sentiment verisiyle fine-tune edildi. |
| 7 | `05_evaluate_sp500_finetuned_models.ipynb` | Modeller bağımsız S&P 500 anotasyon setinde karşılaştırıldı. |
| 8 | `06_evaluate_synthetic_finetuned_models.ipynb` | Modeller dengeli sentetik finans haberleri üzerinde test edildi. |
| 9 | `07_evaluate_unfinetuned_models.ipynb` | Fine-tune edilmemiş taban modellerin performansı kontrol edildi. |
| 10 | `08_evaluate_reuters_finetuned_models.ipynb` | Modeller Reuters 5.000 anotasyon setinin tamamında test edildi. |

## 4. Eğitim, Validation ve Test Bölünmesi

| Bölüm | Toplam | Negative | Neutral | Positive |
| --- | ---: | ---: | ---: | ---: |
| Train | 12.950 | 1.820 | 8.137 | 2.993 |
| Validation | 1.432 | 215 | 929 | 288 |
| Test | 2.386 | 358 | 1.549 | 479 |

## 5. İç Test Sonuçları

Plain sentiment veri setinden ayrılan `2.386` satırlık test bölümü üzerinde elde
edilen sonuçlar:

| Sıra | Model | Accuracy | Macro-F1 | Weighted-F1 |
| ---: | --- | ---: | ---: | ---: |
| 1 | **RoBERTa-base** | **0.8906** | **0.8651** | **0.8918** |
| 2 | BERT-base-uncased | 0.8722 | 0.8378 | 0.8733 |
| 3 | DistilBERT-base-uncased | 0.8583 | 0.8218 | 0.8593 |
| 4 | Original FinBERT | 0.7323 | 0.6794 | 0.7400 |

**Yorum:** Hedef veri seti üzerinde yapılan fine-tuning bütün modellerde hazır
FinBERT baseline sonucunu geliştirmiştir. En iyi model RoBERTa-base olmuştur.
RoBERTa, FinBERT'e göre accuracy değerini `0.1583`, macro-F1 değerini ise
`0.1857` artırmıştır.

## 6. Bağımsız S&P 500 Dış Testi

Modeller, eğitim setinden bağımsız `960` S&P 500 haber başlığı üzerinde test
edilmiştir.

| Sıra | Model | Accuracy | Macro-F1 | Weighted-F1 |
| ---: | --- | ---: | ---: | ---: |
| 1 | **RoBERTa-base** | **0.8125** | **0.8138** | **0.8146** |
| 2 | Original FinBERT | 0.7583 | 0.7585 | 0.7579 |
| 3 | DistilBERT-base-uncased | 0.7531 | 0.7523 | 0.7536 |
| 4 | BERT-base-uncased | 0.7521 | 0.7511 | 0.7524 |

**Yorum:** RoBERTa-base dış testte de en iyi sonucu vermiştir. FinBERT'e göre
macro-F1 farkı `+0.0553` seviyesindedir. Bu sonuç, RoBERTa'nın yalnızca iç test
verisini öğrenmediğini ve farklı S&P 500 başlıklarına daha iyi genelleyebildiğini
göstermektedir.

## 7. Sentetik Finans Haberleri Testi

Her sınıftan `1.000` örnek olmak üzere toplam `3.000` sentetik finans haberiyle
dengeli bir değerlendirme yapılmıştır.

| Sıra | Model | Accuracy | Macro-F1 | Weighted-F1 |
| ---: | --- | ---: | ---: | ---: |
| 1 | **RoBERTa-base** | **0.9957** | **0.9957** | **0.9957** |
| 2 | BERT-base-uncased | 0.9743 | 0.9742 | 0.9742 |
| 3 | DistilBERT-base-uncased | 0.9583 | 0.9581 | 0.9581 |
| 4 | Original FinBERT | 0.9550 | 0.9551 | 0.9551 |

**Yorum:** Tüm modeller sentetik sette yüksek başarı göstermiştir. Sentetik
metinler sınıfları daha açık ifadelerle temsil ettiği için bu sonuçlar gerçek
haber testleriyle birlikte değerlendirilmelidir. Bu sette de RoBERTa-base ilk
sıradadır.

## 8. Reuters Dış Testleri

### Reuters 5.000 Ön Değerlendirmesi

| Sıra | Model | Accuracy | Macro-F1 | Weighted-F1 |
| ---: | --- | ---: | ---: | ---: |
| 1 | Original FinBERT file label | 0.6966 | 0.6866 | 0.7016 |
| 2 | **RoBERTa-base** | **0.6268** | **0.6388** | **0.6486** |
| 3 | BERT-base-uncased | 0.6034 | 0.6131 | 0.6240 |
| 4 | DistilBERT-base-uncased | 0.5766 | 0.5869 | 0.5953 |

**Yorum:** Reuters testinde hazır FinBERT modeli daha başarılıdır. Bu durum,
model başarısının veri kaynağına ve etiketleme mantığına bağlı olduğunu
göstermektedir. RoBERTa, hedef plain sentiment verisinde ve S&P 500 başlıklarında
daha güçlü iken; FinBERT, klasik finans haber diline daha yakın Reuters
başlıklarında avantajını korumuştur.

## 9. Genel Sonuç

| Test Alanı | En Başarılı Model | Accuracy | Macro-F1 | İkinci Model | Accuracy | Macro-F1 |
| --- | --- | ---: | ---: | --- | ---: | ---: |
| Plain sentiment iç test | RoBERTa-base | 0.8906 | 0.8651 | BERT-base-uncased | 0.8722 | 0.8378 |
| Bağımsız S&P 500 dış test | RoBERTa-base | 0.8125 | 0.8138 | Original FinBERT | 0.7583 | 0.7585 |
| Sentetik finans haberleri | RoBERTa-base | 0.9957 | 0.9957 | BERT-base-uncased | 0.9743 | 0.9742 |
| Reuters 5.000 ön değerlendirmesi | Original FinBERT file label | 0.6966 | 0.6866 | RoBERTa-base | 0.6268 | 0.6388 |

Sonuç olarak tek bir model bütün veri kaynaklarında mutlak üstünlük
sağlamamıştır. RoBERTa-base hedef veri setine uyum ve S&P 500 genellemesi
açısından en güçlü modeldir. Reuters ön değerlendirmesinde FinBERT'in öne geçmesi ise alan
uyumunun ve etiket tanımının model performansını doğrudan etkilediğini
göstermektedir.
